# Phase 2 — Grade claims & tag severity (CPU + OpenAI)

Runs on the **same Kaggle session as Phase 1** (no GPU needed). For each cached claim: an OpenAI judge grades it against the K-QA gold statements (`0`=true, `1`=hallucination, `-1`=unverifiable) and tags a severity tier (`dangerous`/`benign`). Output is cached to `claims_phase2.jsonl`.

### Before running
1. Run **Phase 1 first** in this session so `/kaggle/working/claims_phase1.jsonl` and `/kaggle/working/kqa.jsonl` exist. (Fresh session? Re-run Phase 1, or add them as inputs and fix the paths below.)
2. Make sure the **code dataset** (the `sac/` folder) is still added via **+ Add Input** — same `SAC_PATH` as Phase 1.
3. Add an **`OPENAI_API_KEY`** secret: Add-ons → Secrets. (Never paste the key into a cell.)

In [ ]:
!pip -q install -U openai

# ---------- Config ----------
SAC_PATH  = "/kaggle/input/severity-aware-conformal"   # folder that contains the sac/ package
KQA       = "/kaggle/working/kqa.jsonl"                # written by Phase 1
CACHE_IN  = "/kaggle/working/claims_phase1.jsonl"      # written by Phase 1
CACHE_OUT = "/kaggle/working/claims_phase2.jsonl"      # graded claims land here
MODEL     = "gpt-4o"
# ----------------------------

import os, sys
sys.path.insert(0, SAC_PATH)

from kaggle_secrets import UserSecretsClient
os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY")

from sac.kqa_loader import load_kqa, gold_statements
from sac.cache import load_claims, append_claims, existing_claim_ids
from sac.grader import grade_claim, tag_severity
from sac.openai_judge import OpenAIJudge

## 1. Load inputs + build the judge

In [ ]:
items  = {it.qid: it for it in load_kqa(KQA)}   # qid -> KQAItem (gold statements)
claims = load_claims(CACHE_IN)
judge  = OpenAIJudge(model=MODEL)
done   = existing_claim_ids(CACHE_OUT)          # resume support
print(f"{len(claims)} claims to grade | {len(done)} already done")

## 2. Grade + tag severity  (checkpointed, resumable)

In [ ]:
from tqdm.auto import tqdm

pending = [c for c in claims if c.claim_id not in done]   # bar + ETA reflect real work
for c in tqdm(pending, desc="grading"):
    statements = gold_statements(items[c.answer_id])
    c.label, c.grader_rationale    = grade_claim(c.text, statements, judge)
    c.tier,  c.severity_rationale  = tag_severity(c.text, judge)
    append_claims(CACHE_OUT, [c])                         # checkpoint after each claim
    done.add(c.claim_id)
print("graded:", len(load_claims(CACHE_OUT)))

## 3. Distribution sanity check

In [ ]:
from collections import Counter
graded = load_claims(CACHE_OUT)
print("labels:", Counter(c.label for c in graded))   # expect a mix of 0 / 1 / -1
print("tiers :", Counter(c.tier  for c in graded))   # expect dangerous / benign

# Persist for Phase 3: 'Save Version', or copy /kaggle/working/claims_phase2.jsonl to a Kaggle Dataset.